In [119]:
exams_list = [
    "Curso de Formação - Auditor Fiscal do Trabalho - 2025 - Prova Objetiva 1 - Aplicação 1",
    "Curso de Formação - Auditor Fiscal do Trabalho - 2025 - Prova Objetiva 1 - Aplicação 2",
    "Curso de Formação - Auditor Fiscal do Trabalho - 2025 - Prova Objetiva 2"
]

In [120]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import pdfplumber
import re
import os
from utils.utils_oab import extract_text, find_middle, extract_by_columns, join_texts, find_question_pages, load_rules, clean_text, split_questions, extract_questions
from utils.utils_mte import extract_answer_key_formacao
idx = 2
print(f'Prova: {exams_list[idx]}')
exam_path = f'all_exams_mte/{exams_list[idx]}-prova.pdf'
answer_key_path = f'all_exams_mte/{exams_list[idx]}-gabarito.pdf'
#edicao = exams_list[idx].split("Exame da Ordem Unificado")[1].split(" - ")[0].strip()
size_pdf = 0
with pdfplumber.open(exam_path) as pdf:
    size_pdf = len(pdf.pages)
literals, patterns = load_rules('extraction_config/cleaning_rules_mte_formacao.json')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Prova: Curso de Formação - Auditor Fiscal do Trabalho - 2025 - Prova Objetiva 2


In [121]:
dict_mid = {i: 0.5 for i in range(size_pdf)}
# Column split (fraction of the page width); 1.0 is a page in a single column (whole width)
custom_mid = {
    "Curso de Formação - Auditor Fiscal do Trabalho - 2025 - Prova Objetiva 1 - Aplicação 1": {9: 1.0},
    "Curso de Formação - Auditor Fiscal do Trabalho - 2025 - Prova Objetiva 1 - Aplicação 2": {9: 1.0},
}
dict_mid.update(custom_mid.get(exams_list[idx], {}))
meio = find_middle(exam_path, dict_mid)

In [122]:
aa_left, aa_right = extract_by_columns(exam_path, dict_mid)

In [123]:
aa_right

['CEBRASPE – MTE –Edital: 2025\ncorreta, de acordo com o respectivo comando. Para as devidas\npara a correção das suas respostas.\nà luz das apostilas do Curso de Formação, deixando-se de parte\n“Espaço livre” — que constarem deste caderno de prova poderão\nJETIVA 2 --\nQuestão 4\nO nexo técnico epidemiológico previdenciário é aplicável\nquando houver significância estatística entre\nI patologias e exposições constantes das listas A e B do\nanexo II do Decreto n.º 3.048/1999.\nII o código da classificação internacional de doenças (CID) e o\nda classificação nacional de atividade econômica (CNAE), na\nparte inserida pelo Decreto n.º 6.042/2007, na lista B do\nanexo II do Decreto n.º 3.048/1999.\nIII o código da classificação internacional de doenças – CID –\ncom a classificação brasileira de ocupações – CBO.\nAssinale a opção correta.\nA Apenas o item I está certo.\nB Apenas o item II está certo.\nC Apenas os itens I e III estão certos.\nD Apenas os itens II e III estão certos.\nE Todos

In [124]:
questions = join_texts(aa_left, aa_right)

In [125]:
questions = clean_text(questions, literals, patterns)

In [126]:
questions_split = split_questions(questions)

In [127]:
question_literals, question_patterns = load_rules('extraction_config/cleaning_rules_mte_formacao.json', stage='question')
list_questions = extract_questions(questions_split, question_literals, question_patterns, n_options=5)

In [128]:
extracted_questions = extract_questions(questions_split, question_literals, question_patterns, n_options=5)

In [129]:
extracted_questions

[{'question_number': 1,
  'question': 'Caso certa empresa de fabricação de pneus, ao expandir sua linha de produção, decida adquirir novos maquinários e, devido a esses novos equipamentos, passe a utilizar diversos produtos químicos diferentes dos atuais, então, no sentido de mitigar os possíveis riscos ocupacionais que possam surgir frente às mudanças planejadas, essa empresa deverá fazer estudo de',
  'options': {'a': 'avaliação quantitativa.',
   'b': 'controle.',
   'c': 'antecipação.',
   'd': 'reconhecimento.',
   'e': 'avaliação qualitativa.'}},
 {'question_number': 2,
  'question': 'Durante a fiscalização de uma obra de construção civil, foi verificado que trabalhadores não utilizavam os devidos EPIs nas atividades de corte de madeira e de telhas de fibrocimento. Com base na situação precedente, assinale a opção em que estão apresentados os aerodispersóides aos quais os trabalhadores estavam sendo expostos.',
  'options': {'a': 'fibra de madeira e fumos de amianto',
   'b': 'po

In [130]:
gabarito_text = []
with pdfplumber.open(answer_key_path) as pdf:
    for pagina in pdf.pages:
        gabarito_text.append(pagina.extract_text())

In [131]:
answer_key = extract_answer_key_formacao(answer_key_path)
if len(answer_key) != len(extracted_questions):
    print(f'Answer key has {len(answer_key)} answers, but {len(extracted_questions)} questions were extracted.')

In [132]:
# answer is None for annulled questions
for question_data in extracted_questions:
    question_data['answer'] = answer_key.get(question_data['question_number'])

In [133]:
csv_path = 'csvs/mte/mte_formacao_questions_all.csv'
df_exam = pd.DataFrame([{
    'question_number': question_data['question_number'],
    'question': question_data['question'],
    'options': list(question_data['options'].values()),
    'exam': exams_list[idx],
    'answer': question_data['answer'],
} for question_data in extracted_questions])

if os.path.exists(csv_path):
    df_saved = pd.read_csv(csv_path, encoding='utf-8-sig')
    df_saved = df_saved[df_saved['exam'] != exams_list[idx]]
    df_exam = pd.concat([df_saved, df_exam], ignore_index=True)
df_exam.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'{csv_path}: {len(df_exam)} questions from {df_exam["exam"].nunique()} exams')

csvs/mte/mte_formacao_questions_all.csv: 180 questions from 3 exams
